# GNN training
Single train/validation/test split **or** k-fold cross-validation, with the same training loop (see *Training mode and database* below). Replaces `train-GNN.ipynb` + `train-GNN-kfold.ipynb`.

# Load libraries and set device up

In [ ]:
import numpy    as np
import torch.nn as nn
import os
import shutil
import torch

import libraries.model   as clm
import libraries.dataset as cld

from torch_geometric.loader import DataLoader

# Checking if pytorch can run in GPU, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
target_folder = 'model'
os.makedirs(target_folder, exist_ok=True)

# Training mode and database

* `n_splits = 0`: single train / validation / test split (as before).
* `n_splits >= 2`: grouped k-fold cross-validation. Whole materials are assigned to folds. Each run trains **one fold** (`fold_idx_to_run`), with the held-out fold as *test* set and a slice of the training materials as *validation* (early stopping). Restart the kernel (or run with papermill) and increase `fold_idx_to_run`; when all folds are done run the **K-fold summary** cells at the end. To get a deployable model afterwards, run once more with `n_splits = 0`: its uncertainty reference then uses the out-of-fold residuals.
* `database`: any database in the layout read by `cld.generate_dataset` (Pheasy-like: one folder per entry with a structure and a free-energy table or phonon DOS). Changing it **and deleting `model/dataset.pt`** retrains on a custom database.
* `pretrained_folder`: fine-tune a model (`model.pt` + `standardized_parameters.json` in that folder, trained with this version of `libraries/model.py`) on the custom database. The pretrained standardization is reused. Use a smaller `learning_rate`.

In [ ]:
n_splits        = 0   # 0: single split; >= 2: number of folds of the cross-validation
fold_idx_to_run = 0   # k-fold only: fold used as test set in this run (0 ... n_splits - 1)

database           = 'datasets/pheasy'          # database used to generate the training targets
training_temperatures = np.arange(100, 701, 50)  # temperatures (K) of the training targets

pretrained_folder = None  # e.g. 'model-baseline' to fine-tune; None trains from scratch

assert n_splits == 0 or (n_splits >= 2 and 0 <= fold_idx_to_run < n_splits), 'fold_idx_to_run must be in [0, n_splits)'

In [ ]:
# k-fold runs are namespaced by the number of folds and the fold index, so different folds never overwrite each other
tag = '' if n_splits == 0 else f'n{n_splits}_fold{fold_idx_to_run}_'

files_names = {
    'dataset':            f'{target_folder}/dataset.pt',  # raw dataset (shared by all folds)
    'train_dataset_std':  f'{target_folder}/{tag}train_dataset_std.pt',
    'val_dataset_std':    f'{target_folder}/{tag}val_dataset_std.pt',
    'test_dataset_std':   f'{target_folder}/{tag}test_dataset_std.pt',
    'r_dataset_std':      f'{target_folder}/ref_dataset_std.pt',
    'std_parameters':     f'{target_folder}/{tag}standardized_parameters.json',
    'dataset_parameters': f'{target_folder}/dataset_parameters.json',
    'uncertainty_data':   f'{target_folder}/uncertainty_data.json',
    'model':              f'{target_folder}/{tag}model.pt',
    'model_parameters':   f'{target_folder}/{tag}model_parameters.json',
    'oof':                f'{target_folder}/{tag}oof.json',  # k-fold: out-of-fold predictions of this fold
    'oof_residuals':      f'{target_folder}/oof_residuals.json'  # k-fold: residuals of all folds (see the end)
}
cld.save_json(files_names, f'{target_folder}/files_names.json')

# Define parameters

In [ ]:
n_epochs =      200
batch_size =    128
learning_rate = 0.0001
dropout =       0.2
patience =      n_epochs
delta =         2
train_ratio =   0.8
test_ratio =    0.1  # val_ratio = 1 - train_ratio - test_ratio
kfold_val_ratio = 0.1  # k-fold: share of the training materials used as validation (early stopping)

model_parameters = {
    'n_epochs':      n_epochs,
    'batch_size':    batch_size,
    'learning_rate': learning_rate,
    'dropout':       dropout,
    'patience':      patience,
    'delta':         delta,
    'train_ratio':   train_ratio,
    'test_ratio':    test_ratio,
    'n_splits':      n_splits,
    'kfold_val_ratio': kfold_val_ratio
}
cld.save_json(model_parameters, files_names['model_parameters'])

# Generate or load graph database for training

In [ ]:
def standardize_all(train_dataset, val_dataset, test_dataset):
    """Standardize with the training statistics, or with the parameters of the pretrained model when fine-tuning."""
    if pretrained_folder is None:
        train_dataset_std, standardized_parameters = cld.standardize_dataset(train_dataset, transformation='inverse-quadratic')
    else:
        standardized_parameters = cld.load_json(f'{pretrained_folder}/standardized_parameters.json')
        train_dataset_std = cld.standardize_dataset_from_keys(train_dataset, standardized_parameters)

    # Standardize test and validation datasets with train parameters
    val_dataset_std  = cld.standardize_dataset_from_keys(val_dataset,  standardized_parameters)
    test_dataset_std = cld.standardize_dataset_from_keys(test_dataset, standardized_parameters)
    return train_dataset_std, val_dataset_std, test_dataset_std, standardized_parameters


# Standardized datasets are cached only for the single split; each k-fold fold is rebuilt from the raw dataset
cached = n_splits == 0 and all(os.path.exists(files_names[key]) for key in
                               ('train_dataset_std', 'val_dataset_std', 'test_dataset_std', 'std_parameters'))

if cached:
    train_dataset_std, val_dataset_std, test_dataset_std, standardized_parameters = cld.load_datasets(files_names)

else:
    if not os.path.exists(files_names['dataset']):
        # Generate data
        cld.generate_dataset(database,
                             targets=['free-energy'],
                             temperatures=training_temperatures,
                             data_folder=target_folder)

    # Load the raw dataset, with corresponding labels, and standardize it
    dataset = torch.load(files_names['dataset'], weights_only=False)
    
    # Split datasets
    if n_splits == 0:
        train_dataset, val_dataset, test_dataset = cld.split_dataset(train_ratio, test_ratio, dataset, target_folder)
    else:
        train_dataset, val_dataset, test_dataset = cld.split_dataset_kfold(n_splits, fold_idx_to_run, dataset, target_folder,
                                                                           val_ratio=kfold_val_ratio)
    del dataset  # Free up CUDA memory

    # Standardize datasets
    train_dataset_std, val_dataset_std, test_dataset_std, standardized_parameters = standardize_all(train_dataset,
                                                                                                    val_dataset,
                                                                                                    test_dataset)

    # Save datasets
    if n_splits == 0:
        cld.save_datasets(train_dataset_std, val_dataset_std, test_dataset_std, files_names)

    # Save standardized parameters
    cld.save_json(standardized_parameters, files_names['std_parameters'])

# Defining target factor
target_factor = np.array(standardized_parameters['target_std']) / standardized_parameters['scale']
targe_mean    = standardized_parameters['target_mean']

Define data loaders.

In [ ]:
train_loader = DataLoader(train_dataset_std, batch_size=batch_size, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(val_dataset_std,   batch_size=batch_size, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_dataset_std,  batch_size=batch_size, shuffle=False, pin_memory=True)

# Determine number of node-level features in dataset, considering the t_step information
n_node_features = train_dataset_std[0].num_node_features

# Generate Graph Neural Network model

In [ ]:
# Fine-tuning: start from the weights of the pretrained model (load_model reads files_names['model'] if it exists)
if pretrained_folder is not None and not os.path.exists(files_names['model']):
    shutil.copy(f'{pretrained_folder}/model.pt', files_names['model'])

# NOTE: if files_names['model'] already exists it is loaded and the training continues from it (delete it to start from scratch)
if os.path.exists(files_names['model']):
    print(f"Found {files_names['model']}: training continues from it")

model = clm.load_model(n_node_features, dropout, device, model_name=files_names['model'], mode='train')
model

# Train

Define training optimized and criterion

In [ ]:
# MSELoss is by default defined as the mean within the batch
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.MSELoss()

# Initialize early stopping
early_stopping = clm.EarlyStopping(patience=patience, delta=delta, model_name=files_names['model'])

In [ ]:
# Train the model
train_losses = []
val_losses   = []
for epoch in range(n_epochs):
    train_loss, train_predictions, train_ground_truths = clm.train(model, criterion, train_loader,
                                                                   target_factor,
                                                                   standardized_parameters['target_mean'],
                                                                   optimizer)
    val_loss,   val_predictions,   val_ground_truths   =  clm.test(model, criterion, val_loader,
                                                                   target_factor,
                                                                   standardized_parameters['target_mean'])

    # Convert to original units
    train_loss = np.sum(np.sqrt(train_loss) * target_factor)
    val_loss   = np.sum(np.sqrt(val_loss)   * target_factor)

    if epoch%10 == 0:
        cld.parity_plot(train=np.array([train_ground_truths, train_predictions]),
                        validation=np.array([val_ground_truths, val_predictions]))
    
    # Append losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Check early stopping criteria
    early_stopping(val_loss, model)

    if early_stopping.early_stop:
        print('Early stopping')
        break

    print(f'Epoch: {epoch+1}, Train MAE: {train_loss:.4f}, Val MAE: {val_loss:.4f}')

In [ ]:
cld.losses_plot(train_losses=train_losses,
                val_losses=val_losses,
                to_log=True)

# Check test data

In [ ]:
model = clm.load_model(n_node_features, dropout, device, model_name=files_names['model'], mode='eval')
model

In [ ]:
train_loss, train_predictions, train_ground_truths = clm.test(model, criterion, train_loader,
                                                              target_factor,
                                                              standardized_parameters['target_mean'])
val_loss,   val_predictions,   val_ground_truths   = clm.test(model, criterion, val_loader,
                                                              target_factor,
                                                              standardized_parameters['target_mean'])
test_loss,  test_predictions,  test_ground_truths  = clm.test(model, criterion, test_loader,
                                                              target_factor,
                                                              standardized_parameters['target_mean'])

# Pass to energy units (same as initial Fv)
train_loss = np.sum(np.sqrt(train_loss) * target_factor)
val_loss   = np.sum(np.sqrt(val_loss)   * target_factor)
test_loss  = np.sum(np.sqrt(test_loss)  * target_factor)

cld.parity_plot(train=np.array([train_ground_truths, train_predictions]),
                validation=np.array([val_ground_truths, val_predictions]),
                test=np.array([test_ground_truths, test_predictions]),
                save_to=f'{target_folder}/GCNN-training.pdf')

print(f'Train MAE: {train_loss:.4f}, Val MAE: {val_loss:.4f}, Test MAE: {test_loss:.4f}')

In [ ]:
# K-fold: save the out-of-fold predictions of this fold (these materials were never seen during its training)
if n_splits != 0:
    cld.save_json({'fold':         fold_idx_to_run,
                   'test_mae':     float(test_loss),
                   'uid':          cld.get_uids(test_dataset_std, standardized_parameters),
                   'ground_truth': test_ground_truths,
                   'prediction':   test_predictions},
                  files_names['oof'])
    if fold_idx_to_run + 1 < n_splits:
        print(f'Fold {fold_idx_to_run + 1}/{n_splits} done. Restart the kernel, set fold_idx_to_run = {fold_idx_to_run + 1} and run again.')
    else:
        print('All folds done: run the K-fold summary cells at the end of this notebook.')

# Generate uncertainties and save

Single split (`n_splits = 0`) only: this produces the reference dataset and uncertainty data used by the prediction notebook.

In [ ]:
if n_splits == 0:
    # Define reference dataset
    # This is not necessarily just the full dataset, but it can include any
    # additional validation not used as train, test or validation
    dataset = [data for dataset in [train_dataset_std, val_dataset_std, test_dataset_std] for data in dataset]
    uids    = cld.get_uids(dataset, standardized_parameters)
    for data in dataset:
        data.label = f'{data.label} {data.y[0]}'

    # Save reference dataset
    torch.save(dataset, files_names['r_dataset_std'])

    # Generate data loader
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True)


    loss, predictions, ground_truths = clm.test(model, criterion, loader,
                                                target_factor,
                                                standardized_parameters['target_mean'])

    # label: uncertainty = ground-truth - prediction
    uncertainties = ground_truths - predictions

    # If a k-fold was run, use its out-of-fold residuals (each material predicted by models that never saw it)
    # instead of the residuals of this model on the data it was trained on
    if os.path.exists(files_names['oof_residuals']):
        oof_residuals = cld.load_json(files_names['oof_residuals'])
        uncertainties = np.array([oof_residuals.get(uid, residual) for uid, residual in zip(uids, uncertainties)])
        print(f"{sum(uid in oof_residuals for uid in uids)}/{len(uids)} residuals taken from the k-fold")

    # Standardize for better interpolation
    uncert_mean  = uncertainties.mean()
    uncert_std   = uncertainties.std()
    uncert_scale = 1

    uncertainties_std = (uncertainties - uncert_mean) * uncert_scale / uncert_std

    # Check all labels are unique, otherwise you should rename them
    labels = [data.label for data in dataset]
    if len(labels) == len(set(labels)):
        print('All labels are unique')
    else:
        print('Error: some labels are duplicated')

    # Generate dictionary with uncertainty data
    uncertainty_values = {}
    for idx in range(len(dataset)):
        uncertainty_values.update(
            {dataset[idx].label: float(uncertainties_std[idx])}
        )
    uncertainty_data = {
        'uncertainty_values': uncertainty_values,
        'uncert_mean':        uncert_mean,
        'uncert_std':         uncert_std,
        'uncert_scale':       uncert_scale
    }

    # Dump the dictionary with numpy arrays to a JSON file
    cld.save_json(uncertainty_data, files_names['uncertainty_data'])

# K-fold summary

Run once every fold has been trained (set `n_splits` to the number of folds; `fold_idx_to_run` is irrelevant here). It reports the cross-validation error and stores the residuals of every material predicted out-of-fold, which `n_splits = 0` then uses for the uncertainty reference.

In [ ]:
assert n_splits != 0, 'Set n_splits to the number of folds that were trained'

missing = [k for k in range(n_splits) if not os.path.exists(f'{target_folder}/n{n_splits}_fold{k}_oof.json')]
if missing:
    raise RuntimeError(f'Missing results for fold(s) {missing}. Train each one (set fold_idx_to_run) before this cell.')

oof = [cld.load_json(f'{target_folder}/n{n_splits}_fold{k}_oof.json') for k in range(n_splits)]

fold_maes = np.array([o['test_mae'] for o in oof]).flatten()
print(f'K-fold CV test MAE: {fold_maes.mean():.4f} +/- {fold_maes.std():.4f}')
for k, mae in enumerate(fold_maes):
    print(f'  Fold {k + 1}: {mae:.4f}')

ground_truths = np.concatenate([o['ground_truth'] for o in oof])
predictions   = np.concatenate([o['prediction']   for o in oof])
uids          = [uid for o in oof for uid in o['uid']]

# Every material is tested exactly once, by a model that never trained on it
cld.parity_plot(validation=np.array([ground_truths, predictions]), save_to=f'{target_folder}/GCNN-kfold-parity.pdf')

# Residuals in physical units (independent of the standardization of each fold)
cld.save_json({uid: float(truth - prediction) for uid, truth, prediction in zip(uids, ground_truths, predictions)},
              files_names['oof_residuals'])
print(f"Out-of-fold residuals saved to {files_names['oof_residuals']}")